In [ ]:

import numpy as np


class LinearRegression:
    def fit(self, X, y):
        X = np.c_[np.ones(len(X)), X]
        self.w = np.linalg.lstsq(X, y, rcond=None)[0]
        return self

    def predict(self, X):
        return np.c_[np.ones(len(X)), X] @ self.w


import numpy as np
from scipy.special import expit

class LogisticRegression:
    def __init__(self, lr=0.1, epochs=1000):
        self.lr, self.epochs = lr, epochs

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).ravel()

        self.w = np.zeros(X.shape[1])
        self.b = 0

        for _ in range(self.epochs):
            p = expit(X @ self.w + self.b)
            error = p - y

            self.w -= self.lr * (X.T @ error) / len(y)
            self.b -= self.lr * error.mean()

        return self

    def predict(self, X):
        X = np.asarray(X)
        return (expit(X @ self.w + self.b) >= 0.5).astype(int)



class KNNVectorized:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X = np.asarray(X)
        self.y = np.asarray(y).ravel()
        self.classes, self.labels = np.unique(self.y, return_inverse=True)
        return self

    def predict(self, X):
        X = np.asarray(X)

        dist = ((X[:, None] - self.X) ** 2).sum(axis=2)
        idx = np.argsort(dist, axis=1)[:, :self.k]

        labels = self.labels[idx]
        counts = (labels[:, :, None] == np.arange(len(self.classes))).sum(axis=1)

        return self.classes[counts.argmax(axis=1)]


from scipy.stats import norm
class NaiveBayes:
    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        self.classes, labels = np.unique(y, return_inverse=True)

        mask = (labels[:, None] == np.arange(len(self.classes))).astype(float)
        counts = mask.sum(axis=0)

        self.prior = np.log(counts / len(y))
        self.mean = (mask.T @ X) / counts[:, None]
        self.var = (mask.T @ (X ** 2)) / counts[:, None] - self.mean ** 2
        self.var[self.var == 0] = 1e-9  

        return self

    def predict(self, X):
        X = np.asarray(X)

        scores = norm.logpdf(
            X[:, None, :],
            loc=self.mean,
            scale=np.sqrt(self.var)
        )

        scores = scores.sum(axis=2) + self.prior

        return self.classes[scores.argmax(axis=1)]



class PerceptronVectorized:
    def __init__(self, lr=0.1, epochs=100):
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y):
        y = np.asarray(y).ravel()

        self.weights = np.ones(X.shape[1])
        self.bias = 0.0

        for _ in range(self.epochs):
            y_pred = (X @ self.weights + self.bias >= 0).astype(int)
            error = y - y_pred

            self.weights += self.lr * (X.T @ error) / len(y)
            self.bias += self.lr * error.mean()

        return self

    def predict(self, X):
        return (X @ self.weights + self.bias >= 0).astype(int)

